In [1]:
from pathlib import Path
import sqlite3
import tempfile

import pandas as pd

working_directory = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "data" / "raw" / "tables").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project's data/raw/tables directory.")

raw_database = project_root / "data" / "raw" / "airtel.db"


def is_sqlite_database(path):
    if not path.is_file():
        return False
    with path.open("rb") as database_file:
        return database_file.read(16) == b"SQLite format 3\x00"


if is_sqlite_database(raw_database):
    database_path = raw_database
else:
    processed_directory = project_root / "data" / "processed"
    processed_directory.mkdir(parents=True, exist_ok=True)
    database_path = processed_directory / "airtel_from_csv.db"

    if not is_sqlite_database(database_path):
        with tempfile.NamedTemporaryFile(
            prefix="airtel-", suffix=".db", dir=processed_directory, delete=False
        ) as temporary_database:
            temporary_path = Path(temporary_database.name)

        try:
            build_connection = sqlite3.connect(temporary_path)
            try:
                for csv_path in sorted((project_root / "data" / "raw" / "tables").glob("*.csv")):
                    for chunk in pd.read_csv(csv_path, chunksize=50_000):
                        chunk.to_sql(
                            csv_path.stem,
                            build_connection,
                            if_exists="append",
                            index=False,
                        )
            finally:
                build_connection.close()
            temporary_path.replace(database_path)
        finally:
            if temporary_path.exists():
                temporary_path.unlink()

con = sqlite3.connect(database_path)


def q(sql):
    return pd.read_sql(sql, con)

In [ ]:
q(""" SELECT * FROM customers;""")

In [ ]:
q(""" SELECT * FROM plans;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,daily_data_gb,voice,sms_per_day,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,0.50,Limited,50,NaN,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,1.00,Unlimited,100,NaN,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,1.00,Unlimited,100,NaN,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,1.50,Unlimited,100,NaN,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,2.50,Unlimited,100,NaN,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,2.50,Unlimited,100,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,1.50,Unlimited,100,NaN,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,2.00,Unlimited,100,NaN,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,2.00,Unlimited,100,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,1.33,Unlimited,100,OTT-Lite,1


In [ ]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT circle_id) AS unique_circles
       FROM circles;""")

,total_rows,unique_circles
0,22,22


In [ ]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT plan_id) AS unique_plans
       FROM plans;""")

,total_rows,unique_plans
0,14,14


In [ ]:
q(""" SELECT COUNT(*)  AS total_rows,
       COUNT(DISTINCT customer_id) AS unique_customers
       FROM customers;""")

,total_rows,unique_customers
0,22000,22000


In [ ]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT usage_id) AS unique_usages
         FROM usage_daily;""")

,total_rows,unique_usages
0,1326739,1326739


In [ ]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT recharge_id) AS unique_recharges
         FROM recharges;""")

,total_rows,unique_recharges
0,408089,408089


In [ ]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT complaint_id) AS unique_complaints
            FROM complaints;""")

,total_rows,unique_complaints
0,26841,26841


In [ ]:
q(""" SELECT COUNT(*)
FROM circles c
JOIN customers cu
ON c.circle_id = cu.circle_id;""")

,COUNT(*)
0,22000


In [ ]:
q(""" SELECT COUNT(*)
FROM plans c
JOIN customers cu
ON c.plan_id = cu.current_plan_id;""")

,COUNT(*)
0,22000


In [ ]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN recharges cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,408089


In [ ]:
q(""" SELECT COUNT(*)
FROM plans c
JOIN recharges cu
ON c.plan_id = cu.plan_id;""")

,COUNT(*)
0,408089


In [ ]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN usage_daily cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,1326739


In [ ]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN complaints cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,26841


In [ ]:
q(""" SELECT SUM(is_churned = 1 AND churn_date IS NOT NULL) AS left_dated,
       SUM(is_churned = 0 AND churn_date IS NULL) AS stayed_blank
       FROM customers;""")

,left_dated,stayed_blank
0,5226,16774


In [ ]:
q(""" SELECT SUM(is_resolved = 1 AND resolution_days IS NOT NULL) AS left_dated,
       SUM(is_resolved = 0 AND resolution_days IS NULL) AS stayed_blank
       FROM complaints;""")

,left_dated,stayed_blank
0,18820,8021


In [ ]:
# find the subscribers with no row in the usage_daily table

q(""" SELECT c.customer_id, c.connection_type,
       (SELECT COUNT(*)
        FROM recharges r
        WHERE r.customer_id = c.customer_id) AS recharge_count
FROM customers c
WHERE   not EXISTS (SELECT 1
                    FROM usage_daily u
                    WHERE u.customer_id = c.customer_id);""")

,customer_id,connection_type,recharge_count
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [ ]:
# find each subscriber's last day of usage

q("""WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;""")

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0
